# Chapter 6 — Fine-tuning for classification

Chapter 5 produced a reusable GPT architecture and loaded pretrained OpenAI weights. This chapter adapts those representations to a
binary supervised task: classifying SMS messages as legitimate (`ham`) or unwanted (`spam`).

For a modern instruction-tuned LLM, prompting should normally be evaluated before task-specific fine-tuning. This GPT-2-sized model is
not instruction-tuned, however, and the chapter's main purpose is pedagogical: learning dataset preparation, classification heads,
parameter freezing, supervised loss, and evaluation. A production spam system should still be compared with simpler baselines such as
TF–IDF plus logistic regression.

## 6.1 Downloading the SMS Spam Collection

The UCI dataset contains 5,572 labeled SMS messages in a small ZIP archive. Keep acquisition idempotent: if the final TSV already exists,
skip network and extraction work so rerunning the notebook does not overwrite local data.

In [1]:
import os
import urllib.request
import zipfile
from pathlib import Path

url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
zip_path = "sms_spam_collection.zip"
extracted_path = "sms_spam_collection"
data_file_path = Path(extracted_path) / "SMSSpamCollection.tsv"


def download_and_unzip_spam_data(
    url: str,
    zip_path: str | Path,
    extracted_path: str | Path,
    data_file_path: Path,
) -> None:
    """Download and extract the UCI SMS Spam Collection when absent.

    Args:
        url: URL of the source ZIP archive.
        zip_path: Local path used for the downloaded archive.
        extracted_path: Directory that receives extracted archive members.
        data_file_path: Final path of the renamed tab-separated dataset.

    Returns:
        None.

    Raises:
        urllib.error.URLError: If the dataset download fails.
        zipfile.BadZipFile: If the downloaded archive is invalid.
        OSError: If writing, extracting, or renaming a file fails.
    """
    if data_file_path.exists():
        print(f"{data_file_path} already exists. Skipping download and extraction.")
        return

    # The archive is small enough to download into memory before writing.
    with urllib.request.urlopen(url) as response:
        with open(zip_path, "wb") as out_file:
            out_file.write(response.read())

    # The URL is a trusted UCI source; extract its members into one directory.
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extracted_path)

    # Give the extensionless source file a .tsv suffix that describes its format.
    original_file_path = Path(extracted_path) / "SMSSpamCollection"
    os.rename(original_file_path, data_file_path)
    print(f"File downloaded and saved as {data_file_path}")


download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path)

sms_spam_collection\SMSSpamCollection.tsv already exists. Skipping download and extraction.


## 6.2 Loading messages into a dataframe

The source is tab-separated and has no header. Assign `Label` to the external `ham`/`spam` category and `Text` to the raw message. Each
row is one supervised example; the two columns are its target and input.

In [2]:
import pandas as pd

# df: (num_messages=5_572, num_columns=2)
# The source file has no header, so assign descriptive column names.
df = pd.read_csv(
    data_file_path,
    sep="	",
    header=None,
    names=["Label", "Text"],
)
df

,Label,Text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


### Inspecting class balance

The original dataset contains many more ham messages than spam messages. A classifier that favors ham could therefore obtain deceptively
high raw accuracy without learning useful spam detection.

In [3]:
# Count examples before balancing to expose the original class imbalance.
print(df["Label"].value_counts())

Label
ham     4825
spam     747
Name: count, dtype: int64


## 6.3 Creating a balanced pedagogical dataset

Keep all 747 spam examples and randomly select 747 ham examples. The fixed seed makes the selected subset reproducible.

Downsampling makes accuracy easier to interpret and reduces training cost, but discards most legitimate messages and changes class
prevalence. It is a teaching simplification rather than a universally preferred production strategy.

In [4]:
def create_balanced_dataset(df: pd.DataFrame) -> pd.DataFrame:
    """Downsample ham messages to match the number of spam messages.

    Args:
        df: SMS examples with `Label` and `Text` columns.

    Returns:
        Dataframe containing every spam example and an equally sized,
        reproducibly sampled ham subset.
    """
    # num_spam is the minority-class row count: 747 for this dataset.
    num_spam = df[df["Label"] == "spam"].shape[0]
    # Keep a reproducible subset instead of allowing the majority class to dominate.
    ham_subset = df[df["Label"] == "ham"].sample(
        num_spam,
        random_state=123,
    )
    # balanced_df: (2 * num_spam, num_columns=2)
    balanced_df = pd.concat([ham_subset, df[df["Label"] == "spam"]])
    return balanced_df


balanced_df = create_balanced_dataset(df)
print(balanced_df["Label"].value_counts())

Label
ham     747
spam    747
Name: count, dtype: int64


### Encoding class labels

Cross-entropy expects integer class targets rather than strings. Map legitimate messages to `0` and spam messages to `1`; these meanings
must remain consistent through datasets, training, evaluation, and inference.

In [5]:
# Convert external string labels into integer targets for cross-entropy.
# ham -> 0 and spam -> 1; balanced_df remains (num_messages=1_494, 2).
balanced_df["Label"] = balanced_df["Label"].map({"ham": 0, "spam": 1})

## 6.4 Creating training, validation, and test splits

Shuffle once with a fixed seed, then allocate 70% for parameter updates, 10% for model-selection feedback, and the remaining 20% for a
final held-out estimate. The test set must not guide training decisions.

Integer boundaries can cause a one-row rounding difference from the exact percentages. All rows still belong to exactly one split.

In [6]:
def random_split(
    df: pd.DataFrame,
    train_frac: float,
    validation_frac: float,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Shuffle and partition examples into training, validation, and test sets.

    Args:
        df: Labeled examples to partition.
        train_frac: Fraction assigned to the training set.
        validation_frac: Fraction assigned to the validation set. The remaining
            fraction becomes the test set.

    Returns:
        Training, validation, and test dataframes in that order.
    """
    # Shuffle reproducibly so the concatenated classes are mixed before slicing.
    shuffled_df = df.sample(frac=1, random_state=123).reset_index(drop=True)
    train_end = int(len(shuffled_df) * train_frac)
    validation_end = train_end + int(len(shuffled_df) * validation_frac)

    # With 70%/10%, the unspecificed final 20% becomes the held-out test set.
    train_df = shuffled_df[:train_end]
    validation_df = shuffled_df[train_end:validation_end]
    test_df = shuffled_df[validation_end:]

    return train_df, validation_df, test_df


train_df, validation_df, test_df = random_split(
    balanced_df,
    train_frac=0.7,
    validation_frac=0.1,
)
train_df.shape, validation_df.shape, test_df.shape

((1045, 2), (149, 2), (300, 2))

### Saving reproducible split files

Write each dataframe to a separate CSV. Omitting the pandas index prevents an unrelated numeric column from becoming accidental model
input when the files are loaded later.

In [7]:
# Persist each split without a redundant dataframe-index column.
train_df.to_csv("train.csv", index=False)
validation_df.to_csv("validation.csv", index=False)
test_df.to_csv("test.csv", index=False)

## 6.5 Reusing the GPT-2 tokenizer

Classification inputs must use the same token-to-ID mapping as pretraining. GPT-2's `<|endoftext|>` marker has a dedicated vocabulary
ID when explicitly permitted through `allowed_special`.

The next dataset stage can use this known token as a separator or padding value without expanding `vocab_size`. Encoding returns a Python
list containing one token ID; tensors and batches will be constructed later.

In [8]:
import tiktoken

# Reuse the tokenizer whose vocabulary matches the pretrained GPT-2 weights.
tokenizer = tiktoken.get_encoding("gpt2")
# One special end-of-text marker encodes as (num_tokens=1,).
endoftext_id = tokenizer.encode(
    "<|endoftext|>",
    allowed_special={"<|endoftext|>"},
)
print(endoftext_id)

[50256]


## 6.6 Building a fixed-length classification dataset

PyTorch batches require examples with compatible shapes, but SMS messages contain different token counts. `SpamDataset` therefore:

1. loads one saved split;
2. tokenizes every message with GPT-2;
3. chooses or accepts a shared `max_length`;
4. truncates messages that exceed that length;
5. pads shorter messages with GPT-2's end-of-text token ID; and
6. returns each token-ID tensor with its integer class label.

```text
one input       (num_tokens=max_length,)
one label       ()
complete batch  (batch_size, num_tokens=max_length)
batch labels    (batch_size,)
```

When training, validation, and test datasets are created, they should receive the same explicit `max_length`—normally derived from the
training set and capped at the model's `context_length`. Deriving it independently for each split would produce incompatible shapes and
allow validation or test data to influence preprocessing.

In [9]:
import torch
from torch.utils.data import Dataset


class SpamDataset(Dataset[tuple[torch.Tensor, torch.Tensor]]):
    """Tokenize, truncate, and pad labeled SMS messages from one CSV split."""

    def __init__(
        self,
        csv_file: str | Path,
        tokenizer: tiktoken.Encoding,
        max_length: int | None = None,
        pad_token_id: int = 50256,
    ) -> None:
        """Load one split and prepare fixed-length token-ID sequences.

        Args:
            csv_file: CSV containing `Text` and integer `Label` columns.
            tokenizer: GPT-2 tokenizer used to encode each message.
            max_length: Token IDs retained per message. When `None`, use the
                longest encoded message in this split.
            pad_token_id: Token ID appended to shorter messages. GPT-2 uses
                `50256` for its end-of-text token.
        """
        # data: (num_messages, num_columns=2)
        self.data = pd.read_csv(csv_file)

        # Each inner list initially has its message's natural num_tokens.
        self.encoded_texts = [tokenizer.encode(text) for text in self.data["Text"]]

        if max_length is None:
            # Derive one shared length from the longest message in this split.
            self.max_length = self._longest_encoded_length()
        else:
            self.max_length = max_length
            # Truncation prevents inputs from exceeding the selected capacity.
            self.encoded_texts = [
                encoded_text[: self.max_length] for encoded_text in self.encoded_texts
            ]

        # Every inner list now has num_tokens=max_length, enabling batching.
        self.encoded_texts = [
            encoded_text + [pad_token_id] * (self.max_length - len(encoded_text))
            for encoded_text in self.encoded_texts
        ]

    def __getitem__(
        self,
        index: int,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """Return one fixed-length input and its scalar class label.

        Args:
            index: Zero-based row position in this dataset split.

        Returns:
            Input token IDs shaped `(num_tokens=max_length,)` and a scalar
            class-label tensor shaped `()`.

        Raises:
            IndexError: If `index` is outside the dataset.
        """
        encoded = self.encoded_texts[index]
        label = self.data.iloc[index]["Label"]
        # encoded_tensor: (num_tokens=max_length,); label_tensor: ()
        encoded_tensor = torch.tensor(encoded, dtype=torch.long)
        label_tensor = torch.tensor(label, dtype=torch.long)
        return encoded_tensor, label_tensor

    def __len__(self) -> int:
        """Return the number of labeled messages in this split.

        Returns:
            Number of rows loaded from the CSV file.
        """
        return len(self.data)

    def _longest_encoded_length(self) -> int:
        """Find the largest natural token count in this split.

        Returns:
            Maximum encoded-message length, or `0` for an empty split.
        """
        max_length = 0
        for encoded_text in self.encoded_texts:
            encoded_length = len(encoded_text)
            if encoded_length > max_length:
                max_length = encoded_length
        return max_length

## Chapter 6 summary

The chapter now establishes a reproducible supervised-data pipeline:

- download and inspect the UCI SMS Spam Collection;
- balance the pedagogical dataset and encode ham as `0` and spam as `1`;
- create isolated 70%/10%/20% splits;
- reuse the pretrained GPT-2 tokenizer; and
- truncate and pad every message to a shared fixed length for batching.

Balancing and padding make the learning mechanics approachable, but production evaluation should preserve representative prevalence,
compare simpler baselines, and derive preprocessing decisions from training data alone.

In [10]:
train_dataset = SpamDataset(csv_file="train.csv", max_length=None, tokenizer=tokenizer)

In [11]:
print(train_dataset.max_length)

120


In [12]:
val_dataset = SpamDataset(
    csv_file="validation.csv", max_length=train_dataset.max_length, tokenizer=tokenizer
)
test_dataset = SpamDataset(
    csv_file="test.csv", max_length=train_dataset.max_length, tokenizer=tokenizer
)

In [13]:
from torch.utils.data import DataLoader

num_workers = 0
batch_size = 8
torch.manual_seed(123)

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    drop_last=True,
)
val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    drop_last=False,
)
test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    drop_last=False,
)

In [14]:
for _input_batch, _target_batch in train_loader:
    pass
print("Input batch dimensions:", _input_batch.shape)
print("Label batch dimensions", _target_batch.shape)

Input batch dimensions: torch.Size([8, 120])
Label batch dimensions torch.Size([8])


In [15]:
print(f"{len(train_loader)} training batches")
print(f"{len(val_loader)} validation batches")
print(f"{len(test_loader)} test batches")

130 training batches
19 validation batches
38 test batches


loading pretrained weights into model

In [18]:
CHOOSE_MODEL = "gpt2-small (124M)"
INPUT_PROMPT = "Every effort moves"
BASE_CONFIG = {
    "vocab_size": 50257,
    "context_length": 1024,
    "dropout_rate": 0.0,
    "qkv_bias": True,
    "num_heads": 12,
    "num_layers": 12,
}
model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}
BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

In [19]:
from gpt_download import download_and_load_gpt2

from build_llms_from_scratch_companion.model import GPTConfig, GPTModel
from build_llms_from_scratch_companion.training import load_weights_into_gpt

model_size = CHOOSE_MODEL.split(" ")[-1].lstrip("(").rstrip(")")
settings, params = download_and_load_gpt2(model_size=model_size, models_dir="gpt2")
cfg = GPTConfig(**BASE_CONFIG)
model = GPTModel(cfg)
load_weights_into_gpt(model, params)
model.eval()

File already exists and is up-to-date: gpt2\124M\checkpoint
File already exists and is up-to-date: gpt2\124M\encoder.json
File already exists and is up-to-date: gpt2\124M\hparams.json
File already exists and is up-to-date: gpt2\124M\model.ckpt.data-00000-of-00001
File already exists and is up-to-date: gpt2\124M\model.ckpt.index
File already exists and is up-to-date: gpt2\124M\model.ckpt.meta
File already exists and is up-to-date: gpt2\124M\vocab.bpe


GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(1024, 768)
  (drop_emb): Dropout(p=0.0, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=True)
        (W_key): Linear(in_features=768, out_features=768, bias=True)
        (W_value): Linear(in_features=768, out_features=768, bias=True)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.0, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=7

In [20]:
from build_llms_from_scratch_companion.generation import generate_text_simple
from build_llms_from_scratch_companion.training import text_to_token_ids, token_ids_to_text

text_1 = "Every effort moves you"
token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(text_1, tokenizer),
    max_new_tokens=15,
    context_size=cfg.context_length
)
print(token_ids_to_text(token_ids, tokenizer))

Every effort moves you forward.

The first step is to understand the importance of your work
